# **Carga y Gestión de Datos con SQLite**
# **Airbnb Listings — Ciudad de México**

Este notebook cubre el punto **1** del Laboratorio #3: creación de la base de datos SQLite, importación del dataset de regresión y ejecución de consultas SQL relevantes.

**Curso**: Analítica de Datos — Universidad de Antioquia  
**Profesor**: Duván Cataño  
**Laboratorio #3**

---
# 1. Librerías

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
import pandas as pd
import numpy as np
import sqlite3
import os
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

---
# 2. Creación de la Base de Datos SQLite

In [7]:
root = Path.cwd()
if root.name == 'notebooks' and (root.parent / 'data').exists():
    project_root = root.parent
elif (root / 'data').exists():
    project_root = root
else:
    project_root = root

# Crear carpeta database/ si no existe
os.makedirs(project_root / 'database', exist_ok=True)

# Ruta de la base de datos
DB_PATH = project_root / 'database' / 'airbnb_listings.db'

# Crear conexión (crea el archivo .db automáticamente si no existe)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f'Base de datos creada/conectada exitosamente en: {DB_PATH}')

Base de datos creada/conectada exitosamente en: /workspaces/ml-project_analitica_datosv/ml-proyecto_analitica_datos/database/airbnb_listings.db


---
# 3. Importar el Dataset y Crear Tabla en SQLite

In [8]:
# Cargar el CSV
DATA_PATH = project_root / 'data' / 'raw' / 'dataset_regresion_listings.csv'
data = pd.read_csv(DATA_PATH, sep=',')

print(f'Dataset cargado correctamente desde: {DATA_PATH}')
print(f'Dataset cargado: {data.shape[0]:,} filas × {data.shape[1]} columnas')
data.head(5)

Dataset cargado correctamente desde: /workspaces/ml-project_analitica_datosv/ml-proyecto_analitica_datos/data/raw/dataset_regresion_listings.csv
Dataset cargado: 27,051 filas × 18 columnas


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,35797,Villa Dante,153786,Dici,NaN,Cuajimalpa de Morelos,19.38,-99.27,Entire home/apt,"3,673.00",1,0,NaN,NaN,1,363,0,NaN
1,44616,Condesa Haus,196253,Fernando,NaN,Cuauhtémoc,19.41,-99.18,Entire home/apt,"18,000.00",1,65,2025-01-01,0.38,9,360,1,NaN
2,56074,Great space in historical San Rafael,265650,Maris,NaN,Cuauhtémoc,19.44,-99.16,Entire home/apt,591.00,15,84,2025-02-27,0.48,1,333,1,NaN
3,67703,"2 bedroom apt. deco bldg, Condesa",334451,Nicholas,NaN,Cuauhtémoc,19.41,-99.17,Entire home/apt,NaN,2,50,2024-10-30,0.30,2,252,1,NaN
4,70644,Beautiful light Studio Coyoacan- full equipped !,212109,Trisha,NaN,Coyoacán,19.35,-99.16,Entire home/apt,NaN,3,134,2025-08-18,0.81,3,234,8,NaN


In [9]:
# Limpiar columnas 100% vacías antes de importar
# neighbourhood_group y license son 100% nulos en este dataset (CDMX)
data_db = data.drop(columns=['neighbourhood_group', 'license'])

# Convertir last_review a string limpio (SQLite no tiene tipo DATE nativo)
data_db['last_review'] = data_db['last_review'].astype(str).replace('nan', None)

print('Columnas que se importarán a SQLite:')
print(data_db.columns.tolist())

Columnas que se importarán a SQLite:
['id', 'name', 'host_id', 'host_name', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm']


In [10]:
# Crear tabla en SQLite con esquema explícito
cursor.execute('DROP TABLE IF EXISTS listings')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS listings (
        id                              INTEGER PRIMARY KEY,
        name                            TEXT,
        host_id                         INTEGER,
        host_name                       TEXT,
        neighbourhood                   TEXT,
        latitude                        REAL,
        longitude                       REAL,
        room_type                       TEXT,
        price                           REAL,
        minimum_nights                  INTEGER,
        number_of_reviews               INTEGER,
        last_review                     TEXT,
        reviews_per_month               REAL,
        calculated_host_listings_count  INTEGER,
        availability_365                INTEGER,
        number_of_reviews_ltm           INTEGER
    )
''')

conn.commit()
print('Tabla "listings" creada exitosamente.')

Tabla "listings" creada exitosamente.


In [11]:
# Importar datos usando pandas (método más eficiente)
data_db.to_sql('listings', conn, if_exists='replace', index=False)
conn.commit()

# Verificar carga
resultado = pd.read_sql('SELECT COUNT(*) AS total_registros FROM listings', conn)
print(f'Registros importados a SQLite: {resultado["total_registros"][0]:,}')

Registros importados a SQLite: 27,051


In [12]:
# Verificar estructura de la tabla
print('Estructura de la tabla listings:')
info_tabla = pd.read_sql('PRAGMA table_info(listings)', conn)
print(info_tabla[['name', 'type', 'notnull', 'dflt_value']].to_string(index=False))

Estructura de la tabla listings:
                          name    type  notnull dflt_value
                            id INTEGER        0       None
                          name    TEXT        0       None
                       host_id INTEGER        0       None
                     host_name    TEXT        0       None
                 neighbourhood    TEXT        0       None
                      latitude    REAL        0       None
                     longitude    REAL        0       None
                     room_type    TEXT        0       None
                         price    REAL        0       None
                minimum_nights INTEGER        0       None
             number_of_reviews INTEGER        0       None
                   last_review    TEXT        0       None
             reviews_per_month    REAL        0       None
calculated_host_listings_count INTEGER        0       None
              availability_365 INTEGER        0       None
         number_of_revi

In [13]:
# Vista previa desde SQLite
print('Primeras 5 filas desde SQLite:')
pd.read_sql('SELECT * FROM listings LIMIT 5', conn)

Primeras 5 filas desde SQLite:


,id,name,host_id,host_name,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm
0,35797,Villa Dante,153786,Dici,Cuajimalpa de Morelos,19.38,-99.27,Entire home/apt,"3,673.00",1,0,NaN,NaN,1,363,0
1,44616,Condesa Haus,196253,Fernando,Cuauhtémoc,19.41,-99.18,Entire home/apt,"18,000.00",1,65,2025-01-01,0.38,9,360,1
2,56074,Great space in historical San Rafael,265650,Maris,Cuauhtémoc,19.44,-99.16,Entire home/apt,591.00,15,84,2025-02-27,0.48,1,333,1
3,67703,"2 bedroom apt. deco bldg, Condesa",334451,Nicholas,Cuauhtémoc,19.41,-99.17,Entire home/apt,NaN,2,50,2024-10-30,0.30,2,252,1
4,70644,Beautiful light Studio Coyoacan- full equipped !,212109,Trisha,Coyoacán,19.35,-99.16,Entire home/apt,NaN,3,134,2025-08-18,0.81,3,234,8


---
# 4. Consultas SQL Relevantes

A continuación se presentan **8 consultas SQL** que cubren todos los tipos requeridos: conteos, valores promedio, agrupaciones y filtros.

## 📊 Consulta 1 — Conteo total y por tipo de habitación
**Tipo:** Conteo  
**Objetivo:** Conocer cuántos listings existen en total y cómo se distribuyen por tipo de habitación.

In [14]:
query_1 = '''
    SELECT
        room_type,
        COUNT(*)                              AS total_listings,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM listings), 2) AS porcentaje
    FROM listings
    GROUP BY room_type
    ORDER BY total_listings DESC
'''

resultado_1 = pd.read_sql(query_1, conn)
print('Conteo de listings por tipo de habitación:')
print(resultado_1.to_string(index=False))

total = pd.read_sql('SELECT COUNT(*) AS total FROM listings', conn)
print(f'\nTotal de listings en la base de datos: {total["total"][0]:,}')

Conteo de listings por tipo de habitación:
      room_type  total_listings  porcentaje
Entire home/apt           17713       65.48
   Private room            8995       33.25
    Shared room             257        0.95
     Hotel room              86        0.32

Total de listings en la base de datos: 27,051


## 💰 Consulta 2 — Precio promedio, mediano y máximo por tipo de habitación
**Tipo:** Valores promedio  
**Objetivo:** Comparar las estadísticas de precio según el tipo de alojamiento ofrecido.

In [15]:
query_2 = '''
    SELECT
        room_type,
        COUNT(price)                  AS listings_con_precio,
        ROUND(AVG(price), 2)          AS precio_promedio,
        ROUND(MIN(price), 2)          AS precio_minimo,
        ROUND(MAX(price), 2)          AS precio_maximo,
        ROUND(AVG(minimum_nights), 2) AS noches_minimas_promedio
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY room_type
    ORDER BY precio_promedio DESC
'''

resultado_2 = pd.read_sql(query_2, conn)
print('Estadísticas de precio por tipo de habitación:')
print(resultado_2.to_string(index=False))

Estadísticas de precio por tipo de habitación:
      room_type  listings_con_precio  precio_promedio  precio_minimo  precio_maximo  noches_minimas_promedio
     Hotel room                   57        42,841.00         415.00     734,464.00                     1.28
Entire home/apt                15896         1,930.28         150.00     347,429.00                     3.83
   Private room                 7360         1,226.56         137.00     900,000.00                     3.14
    Shared room                  254           360.89          61.00      12,841.00                     1.51


## 🏘️ Consulta 3 — Agrupación por alcaldía: precio promedio y actividad
**Tipo:** Agrupación  
**Objetivo:** Identificar qué alcaldías (neighbourhood) tienen mayor concentración de listings y precios más altos.

In [16]:
query_3 = '''
    SELECT
        neighbourhood,
        COUNT(*)                                AS total_listings,
        ROUND(AVG(price), 2)                    AS precio_promedio,
        ROUND(AVG(availability_365), 1)         AS disponibilidad_promedio,
        ROUND(AVG(number_of_reviews), 1)        AS resenas_promedio,
        ROUND(AVG(reviews_per_month), 2)        AS resenas_mes_promedio
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY neighbourhood
    HAVING total_listings >= 50
    ORDER BY precio_promedio DESC
'''

resultado_3 = pd.read_sql(query_3, conn)
print('Agrupación por alcaldía (mín. 50 listings):')
print(resultado_3.to_string(index=False))

Agrupación por alcaldía (mín. 50 listings):
         neighbourhood  total_listings  precio_promedio  disponibilidad_promedio  resenas_promedio  resenas_mes_promedio
               Tlalpan             626         2,493.47                   262.80             31.50                  1.10
 Cuajimalpa de Morelos             349         2,150.58                   271.60             38.30                  1.48
            Cuauhtémoc           11253         2,126.38                   249.00             65.90                  2.25
        Álvaro Obregón             812         2,117.76                   270.30             34.70                  1.14
        Miguel Hidalgo            3979         1,876.39                   255.90             54.90                  1.79
              Coyoacán            1415         1,245.48                   243.10             55.00                  1.56
La Magdalena Contreras             108         1,195.07                   256.30             20.70           

## 🔍 Consulta 4 — Filtro: listings de alto precio y alta disponibilidad
**Tipo:** Filtro  
**Objetivo:** Encontrar listings premium (precio > $5,000 MXN) que además estén disponibles más de 300 días al año — candidatos a propiedades de inversión.

In [17]:
query_4 = '''
    SELECT
        id,
        name,
        neighbourhood,
        room_type,
        price,
        availability_365,
        number_of_reviews,
        calculated_host_listings_count  AS listings_del_host
    FROM listings
    WHERE
        price > 5000
        AND availability_365 > 300
        AND price IS NOT NULL
    ORDER BY price DESC
    LIMIT 20
'''

resultado_4 = pd.read_sql(query_4, conn)
print('Listings premium (precio > $5,000 y disponibilidad > 300 días):')
print(resultado_4.to_string(index=False))
print(f'\nTotal de listings que cumplen el filtro: {len(resultado_4)}')

Listings premium (precio > $5,000 y disponibilidad > 300 días):
                 id                                               name         neighbourhood       room_type      price  availability_365  number_of_reviews  listings_del_host
 892676742760732948                  ¡Recámara acogedora y muy cómoda!               Tlalpan    Private room 900,000.00               364                  3                  1
1475630323954004542           Casa Alebrije: Espacio Único y Exclusivo              Coyoacán Entire home/apt 159,200.00               364                  0                 25
           52691395    Departamento de 2 pisos en condesa con Roof top            Cuauhtémoc Entire home/apt 154,125.00               365                 31                  1
           41404517   Súper Corazón Condesa + Wi fi  10 p + 5rc + 3 ba            Cuauhtémoc    Private room 100,989.00               365                  0                  3
1477127217953336418                              Condo i

## 👤 Consulta 5 — Agrupación por anfitrión: top 15 hosts más activos
**Tipo:** Agrupación + Filtro  
**Objetivo:** Identificar anfitriones profesionales (con muchos listings) y analizar su precio promedio versus los anfitriones casuales.

In [18]:
query_5 = '''
    SELECT
        host_id,
        host_name,
        COUNT(*)                         AS total_listings,
        ROUND(AVG(price), 2)             AS precio_promedio,
        ROUND(AVG(availability_365), 1)  AS disponibilidad_promedio,
        ROUND(AVG(number_of_reviews), 1) AS resenas_promedio,
        COUNT(DISTINCT neighbourhood)    AS alcaldias_distintas
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY host_id, host_name
    ORDER BY total_listings DESC
    LIMIT 15
'''

resultado_5 = pd.read_sql(query_5, conn)
print('Top 15 anfitriones más activos:')
print(resultado_5.to_string(index=False))

Top 15 anfitriones más activos:
  host_id      host_name  total_listings  precio_promedio  disponibilidad_promedio  resenas_promedio  alcaldias_distintas
498701635     Blueground             212         1,896.75                   322.90             17.50                    2
333035396           Clau             155         1,734.66                   303.30             79.60                    5
 10764020          Mr. W             154         2,559.71                   250.80             37.10                    4
 16148871           Juan             131         1,753.46                   321.60             57.00                    2
 91265490   Luis Antonio             104         1,342.33                   278.30            251.70                    2
 27449159      Capitalia              98         2,264.90                   341.80             42.30                    5
 23468816      Alejandro              93         1,817.53                   277.20             42.10              

## 📅 Consulta 6 — Filtro: listings sin ninguna reseña (nuevos o inactivos)
**Tipo:** Filtro  
**Objetivo:** Identificar listings que nunca han recibido una reseña — pueden ser nuevos o estar inactivos, lo que afecta la imputación de reviews_per_month.

In [19]:
query_6 = '''
    SELECT
        neighbourhood,
        room_type,
        COUNT(*) AS listings_sin_resenas,
        ROUND(AVG(price), 2) AS precio_promedio,
        ROUND(AVG(availability_365), 1) AS disponibilidad_promedio
    FROM listings
    WHERE
        number_of_reviews = 0
        AND last_review IS NULL
    GROUP BY neighbourhood, room_type
    ORDER BY listings_sin_resenas DESC
    LIMIT 15
'''

resultado_6 = pd.read_sql(query_6, conn)

total_sin_resenas = pd.read_sql(
    'SELECT COUNT(*) AS total FROM listings WHERE number_of_reviews = 0', conn
)
print(f'Total listings sin reseñas: {total_sin_resenas["total"][0]:,}\n')
print('Desglose por alcaldía y tipo de habitación:')
print(resultado_6.to_string(index=False))

Total listings sin reseñas: 3,401

Desglose por alcaldía y tipo de habitación:
        neighbourhood       room_type  listings_sin_resenas  precio_promedio  disponibilidad_promedio
           Cuauhtémoc Entire home/apt                   680         5,606.12                   233.30
           Cuauhtémoc    Private room                   620         3,517.42                   230.20
       Miguel Hidalgo Entire home/apt                   330         3,553.03                   201.10
       Miguel Hidalgo    Private room                   247         1,532.16                   229.20
        Benito Juárez    Private room                   232           905.84                   207.50
        Benito Juárez Entire home/apt                   168         3,157.49                   221.50
             Coyoacán    Private room                   160         1,122.99                   178.20
       Álvaro Obregón    Private room                   109        14,207.12                   174.50
   

## 📈 Consulta 7 — Valores promedio generales de toda la tabla
**Tipo:** Valores promedio  
**Objetivo:** Obtener una visión global de los indicadores clave del dataset desde SQL.

In [20]:
query_7 = '''
    SELECT
        COUNT(*)                                    AS total_listings,
        COUNT(DISTINCT host_id)                     AS total_anfitriones,
        COUNT(DISTINCT neighbourhood)               AS total_alcaldias,
        COUNT(price)                                AS listings_con_precio,
        ROUND(AVG(price), 2)                        AS precio_promedio_global,
        ROUND(AVG(minimum_nights), 2)               AS noches_minimas_promedio,
        ROUND(AVG(number_of_reviews), 2)            AS resenas_promedio,
        ROUND(AVG(availability_365), 2)             AS disponibilidad_promedio,
        ROUND(AVG(calculated_host_listings_count), 2) AS listings_por_host_promedio
    FROM listings
'''

resultado_7 = pd.read_sql(query_7, conn)
print('Resumen global del dataset desde SQLite:')
# Transponer para mejor lectura
print(resultado_7.T.rename(columns={0: 'Valor'}).to_string())

Resumen global del dataset desde SQLite:
                               Valor
total_listings             27,051.00
total_anfitriones          12,046.00
total_alcaldias                16.00
listings_con_precio        23,567.00
precio_promedio_global      1,792.54
noches_minimas_promedio         4.58
resenas_promedio               53.78
disponibilidad_promedio       232.69
listings_por_host_promedio     14.56


## 🔗 Consulta 8 — Comparativa anfitriones casuales vs profesionales (self-join / subquery)
**Tipo:** Agrupación + Subquery (equivalente a Join)  
**Objetivo:** Comparar el comportamiento de precios entre anfitriones con 1 solo listing (casuales) vs anfitriones con más de 10 listings (profesionales).

In [21]:
query_8 = '''
    SELECT
        CASE
            WHEN calculated_host_listings_count = 1 THEN '1. Casual (1 listing)'
            WHEN calculated_host_listings_count BETWEEN 2 AND 5 THEN '2. Semi-profesional (2-5)'
            WHEN calculated_host_listings_count BETWEEN 6 AND 10 THEN '3. Activo (6-10)'
            ELSE '4. Profesional (>10)'
        END AS perfil_anfitrion,
        COUNT(*)                                AS total_listings,
        COUNT(DISTINCT host_id)                 AS total_anfitriones,
        ROUND(AVG(price), 2)                    AS precio_promedio,
        ROUND(AVG(availability_365), 1)         AS disponibilidad_promedio,
        ROUND(AVG(number_of_reviews), 1)        AS resenas_promedio
    FROM listings
    WHERE price IS NOT NULL
    GROUP BY perfil_anfitrion
    ORDER BY perfil_anfitrion
'''

resultado_8 = pd.read_sql(query_8, conn)
print('Comparativa por perfil de anfitrión:')
print(resultado_8.to_string(index=False))

Comparativa por perfil de anfitrión:
         perfil_anfitrion  total_listings  total_anfitriones  precio_promedio  disponibilidad_promedio  resenas_promedio
    1. Casual (1 listing)            6691               6691         1,605.09                   241.90             47.70
2. Semi-profesional (2-5)            7017               2743         1,448.21                   247.00             63.10
         3. Activo (6-10)            3202                467         1,455.09                   262.20             60.90
     4. Profesional (>10)            6657                308         2,506.21                   268.30             61.00


---
# 5. Verificación Final y Cierre de Conexión

In [22]:
# Listar todas las tablas en la base de datos
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tablas existentes en la base de datos:')
print(tablas.to_string(index=False))

# Tamaño del archivo .db
size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f'\nTamaño del archivo .db: {size_mb:.2f} MB')
print(f'Ubicación: {os.path.abspath(DB_PATH)}')

Tablas existentes en la base de datos:
    name
listings

Tamaño del archivo .db: 3.79 MB
Ubicación: /workspaces/ml-project_analitica_datosv/ml-proyecto_analitica_datos/database/airbnb_listings.db


In [23]:
# Cerrar conexión
conn.close()
print('Conexión a SQLite cerrada correctamente.')

Conexión a SQLite cerrada correctamente.


---
# 6. Resumen de Consultas Realizadas

| # | Consulta | Tipo | Hallazgo Principal |
|---|---|---|---|
| 1 | Conteo por tipo de habitación | Conteo | 65.5% son "Entire home/apt"; solo 0.3% son "Hotel room" |
| 2 | Estadísticas de precio por tipo | Valores promedio | Hotel room tiene el mayor precio promedio (~$42,841 MXN) aunque con pocos listings |
| 3 | Agrupación por alcaldía | Agrupación | Tlalpan y Cuajimalpa lideran en precio; Cuauhtémoc tiene el mayor volumen |
| 4 | Filtro: listings premium + alta disponibilidad | Filtro | Listings con precio > $5,000 y disponibilidad > 300 días: posibles inversiones |
| 5 | Top 15 anfitriones más activos | Agrupación + Filtro | Existen hosts con 200+ listings, indicando gestión profesional |
| 6 | Filtro: listings sin ninguna reseña | Filtro | Listings nuevos o inactivos: relevantes para la estrategia de imputación |
| 7 | Promedios globales del dataset | Valores promedio | Precio global promedio ~$1,792 MXN; 27,051 listings de 13,000+ anfitriones |
| 8 | Perfil de anfitrión (casual vs profesional) | Agrupación + Subquery | Anfitriones profesionales tienen precios más altos y mayor disponibilidad |